In [ ]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [ ]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.utils import Logger

#### 1. Загружем датасет с текстами, на основе которого будет построен граф знаний

In [ ]:
# Loading dataset to build graph and other structures 
# TODO

DATASET_PATH = ...
data = ...

#### 2. Задаём конфигурацию графа знаний

In [ ]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='neo4j', db_config=GraphDBConnectionConfig(uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing6', db_name='vectorized_nodes', is_exist=True, need_to_clear=True)), 
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing6', db_name='vectorized_triplets', is_exist=True, need_to_clear=True)),
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='bfs', # TO CHANGE
            retriever_config=BFSSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=KVDBConnectionConfig(
                host='localhost', params={'kvstore_dump_name': 'inmemory_store', 'load_from_disk': False, 'load_dump_dir': '.', # TO CHANGE
                                          'save_on_disk': True, 'save_dump_dir': '.'}))), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig()),
    log=Logger('main_debug'))

#### 3. Инициализируем граф знаний

In [ ]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

In [ ]:
# ATTENTION !!!
rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 4. Добавляем в граф информацию из загрженного датасета

In [ ]:
rkg_main.update_memory(data)

#### 5. Q&A

In [ ]:
examples_questions = []

In [ ]:
rkg_main.answer_question(examples_questions[0])